# 02 — Data quality

These checks cover completeness, key uniqueness, valid domains, and referential
integrity. `days_since_prior_order` is expected to be null for the first order of each
user; that source convention is tested separately instead of being reported as an
error.

In [1]:
from pathlib import Path

import duckdb
import pandas as pd


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "sql" / "create_tables.sql").is_file():
            return candidate
    raise FileNotFoundError("Could not find the project root containing sql/create_tables.sql")


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data" / "raw"
DATABASE_PATH = PROJECT_ROOT / "instacart.duckdb"
PROJECT_ROOT

PosixPath('/Users/sebastiankalita/Documents/DE_portfolio/sql-project/instacart-sql-analysis')

In [2]:
if not DATABASE_PATH.is_file():
    raise FileNotFoundError(
        "Database not found. Run 01_load_and_explore.ipynb or "
        "python scripts/build_database.py first."
    )

## Null profile

In [3]:
tables_to_profile = ["orders", "products", "aisles", "departments", "order_products"]
null_rows = []

with duckdb.connect(str(DATABASE_PATH), read_only=True) as connection:
    for table_name in tables_to_profile:
        columns = connection.execute(
            """
            SELECT column_name
            FROM information_schema.columns
            WHERE table_schema = 'main' AND table_name = ?
            ORDER BY ordinal_position
            """,
            [table_name],
        ).fetchall()
        total_rows = connection.execute(
            f'SELECT COUNT(*) FROM "{table_name}"'
        ).fetchone()[0]
        for (column_name,) in columns:
            null_count = connection.execute(
                f'SELECT COUNT(*) FROM "{table_name}" WHERE "{column_name}" IS NULL'
            ).fetchone()[0]
            null_rows.append(
                {
                    "table_name": table_name,
                    "column_name": column_name,
                    "total_rows": total_rows,
                    "null_count": null_count,
                }
            )

null_profile = pd.DataFrame(null_rows)
null_profile[null_profile["null_count"] > 0]

,table_name,column_name,total_rows,null_count
6,orders,days_since_prior_order,3421083,206209


## Keys, domains, and referential integrity

In [4]:
quality_checks = {
    "duplicate order_id": """
        SELECT COUNT(*) - COUNT(DISTINCT order_id) FROM orders
    """,
    "duplicate product_id": """
        SELECT COUNT(*) - COUNT(DISTINCT product_id) FROM products
    """,
    "duplicate aisle_id": """
        SELECT COUNT(*) - COUNT(DISTINCT aisle_id) FROM aisles
    """,
    "duplicate department_id": """
        SELECT COUNT(*) - COUNT(DISTINCT department_id) FROM departments
    """,
    "duplicate order-product pair": """
        SELECT COUNT(*) - COUNT(DISTINCT (order_id, product_id)) FROM order_products
    """,
    "order_products without order": """
        SELECT COUNT(*) FROM order_products op
        LEFT JOIN orders o USING (order_id)
        WHERE o.order_id IS NULL
    """,
    "order_products without product": """
        SELECT COUNT(*) FROM order_products op
        LEFT JOIN products p USING (product_id)
        WHERE p.product_id IS NULL
    """,
    "products without aisle": """
        SELECT COUNT(*) FROM products p
        LEFT JOIN aisles a USING (aisle_id)
        WHERE a.aisle_id IS NULL
    """,
    "products without department": """
        SELECT COUNT(*) FROM products p
        LEFT JOIN departments d USING (department_id)
        WHERE d.department_id IS NULL
    """,
    "invalid reordered value": """
        SELECT COUNT(*) FROM order_products WHERE reordered NOT IN (0, 1)
    """,
    "invalid add_to_cart_order": """
        SELECT COUNT(*) FROM order_products WHERE add_to_cart_order < 1
    """,
    "invalid order_number": """
        SELECT COUNT(*) FROM orders WHERE order_number < 1
    """,
    "unexpected first-order null pattern": """
        SELECT COUNT(*) FROM orders
        WHERE (order_number = 1 AND days_since_prior_order IS NOT NULL)
           OR (order_number > 1 AND days_since_prior_order IS NULL)
    """,
}

with duckdb.connect(str(DATABASE_PATH), read_only=True) as connection:
    check_results = pd.DataFrame(
        [
            {
                "check": check_name,
                "failing_rows": connection.execute(query).fetchone()[0],
            }
            for check_name, query in quality_checks.items()
        ]
    )

check_results

,check,failing_rows
0,duplicate order_id,0
1,duplicate product_id,0
2,duplicate aisle_id,0
3,duplicate department_id,0
4,duplicate order-product pair,0
5,order_products without order,0
6,order_products without product,0
7,products without aisle,0
8,products without department,0
9,invalid reordered value,0


In [5]:
failed_checks = check_results[check_results["failing_rows"] != 0]
assert failed_checks.empty, f"Data-quality checks failed:\n{failed_checks.to_string(index=False)}"
print(f"All {len(check_results)} data-quality checks passed.")

All 13 data-quality checks passed.
